In [1]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten, Activation
from tensorflow.keras.optimizers import Adam
import random

# ==========================================
# CLASSE: Gerador de Dados Sudoku : (Especificação 4x4)
# ==========================================
class matriz_sudoku:
    """
    Gera tabuleiros de Sudoku 4x4 válidos e cria os conjuntos de dados (X: com buracos, Y: resolvidos).
    Os números de S = {1, 2, 3, 4} serão mapeados para {0, 1, 2, 3} internamente para a rede.
    """
    def __init__(self):
        # Um tabuleiro base válido 4x4
        self.base_board = np.array([
            [0, 1, 2, 3],
            [2, 3, 0, 1],
            [1, 0, 3, 2],
            [3, 2, 1, 0]
        ])

    def generate_valid_boards(self, num_samples):
        boards = []
        for _ in range(num_samples):
            # Clonar o tabuleiro base
            b = self.base_board.copy()

            # Embaralhar blocos lógicos (linhas e colunas dentro dos subgrupos 2x2)
            if random.choice([True, False]): b[[0, 1], :] = b[[1, 0], :]
            if random.choice([True, False]): b[[2, 3], :] = b[[3, 2], :]
            if random.choice([True, False]): b[:, [0, 1]] = b[:, [1, 0]]
            if random.choice([True, False]): b[:, [2, 3]] = b[:, [3, 2]]

            # Permutar os números em S={0, 1, 2, 3} (que representam 1, 2, 3, 4)
            perms = [0, 1, 2, 3]
            random.shuffle(perms)
            for i in range(4):
                for j in range(4):
                    b[i, j] = perms[self.base_board[i, j]]

            boards.append(b)
        return np.array(boards)

    def create_dataset(self, num_samples, min_clues=5, max_clues=10):
        """ Cria tabuleiros com zeros indicando células vazias. """
        y_data = self.generate_valid_boards(num_samples)
        x_data = []

        for board in y_data:
            x_board = board.copy()
            num_clues = random.randint(min_clues, max_clues)
            cells_to_hide = 16 - num_clues

            # Escolher aleatoriamente quais células esconder
            hide_indices = random.sample(range(16), cells_to_hide)
            for idx in hide_indices:
                r, c = divmod(idx, 4)
                x_board[r, c] = -1 # -1 representará 'vazio' para o One-Hot
            x_data.append(x_board)

        return np.array(x_data), y_data

# ==========================================
# FUNÇÕES DE PRÉ-PROCESSAMENTO
# ==========================================
def one_hot_encode(x, y):
    """
    Converte o tabuleiro 4x4 em representação One-Hot para a Rede Neural.
    - Células vazias (representadas por -1) viram vetor de zeros: [0,0,0,0]
    - Valores 0-3 viram [1,0,0,0], [0,1,0,0], etc.
    """
    x_encoded = np.zeros((x.shape[0], 16, 4))
    y_encoded = np.zeros((y.shape[0], 16, 4))

    for i in range(x.shape[0]):
        for j in range(16):
            r, c = divmod(j, 4)
            if x[i, r, c] != -1:
                x_encoded[i, j, x[i, r, c]] = 1
            y_encoded[i, j, y[i, r, c]] = 1

    return x_encoded, y_encoded

def print_board(board, title="Tabuleiro"):
    print(f"--- {title} ---")
    for r in range(4):
        if r == 2: print("-" * 9)
        row_str = ""
        for c in range(4):
            if c == 2: row_str += "| "
            val = board[r, c]
            row_str += str(val + 1) if val != -1 else "."
            row_str += " "
        print(row_str)
    print()

# ==========================================
# VALIDAÇÃO DAS RESTRIÇÕES DO PROBLEMA
# ==========================================
def validate_sudoku(board):
    """
    Verifica se um tabuleiro 4x4 satisfaz:
    - linhas sem repetição
    - colunas sem repetição
    - blocos 2x2 sem repetição
    """

    expected = {0, 1, 2, 3}

    # Linhas
    for row in board:
        if set(row) != expected:
            return False

    # Colunas
    for col in range(4):
        if set(board[:, col]) != expected:
            return False

    # Blocos 2x2
    for br in range(0, 4, 2):
        for bc in range(0, 4, 2):

            block = []

            for r in range(br, br + 2):
                for c in range(bc, bc + 2):
                    block.append(board[r, c])

            if set(block) != expected:
                return False

    return True

# =====================================================
# ARQUITETURA DA REDE NEURAL MULTICAMADAS
# =====================================================
class modelo_sudoku:
    """ Define e treina a RNA Multicamadas """
    def __init__(self):
        self.model = Sequential([
            Flatten(input_shape=(16, 4)),       # Entrada: 16 células x 4 possibilidades (64 valores)
            Dense(128, activation='relu'),      # Camada Oculta 1
            Dense(128, activation='relu'),      # Camada Oculta 2
            Dense(64, activation='relu'),       # Camada Oculta 3
            Dense(16 * 4),                      # Saída linear de 64 valores
            Reshape((16, 4)),                   # Remoldar para 16 células x 4 probabilidades
            Activation('softmax')               # Probabilidade em cada célula (1 a 4)
        ])
        self.model.compile(optimizer=Adam(learning_rate=0.001),
                           loss='categorical_crossentropy',
                           metrics=['accuracy'])

    def train(self, x_train, y_train, epochs=30, batch_size=32):
        print("Iniciando treinamento da RNA...")
        self.model.fit(x_train, y_train, epochs=epochs, batch_size=batch_size, validation_split=0.2, verbose=1)

    def predict(self, x_test_encoded):
        preds = self.model.predict(x_test_encoded)
        # Pega o índice com maior probabilidade (argmax) para cada célula
        return np.argmax(preds, axis=-1).reshape(-1, 4, 4)

# ==========================================
# EXECUÇÃO PRINCIPAL
# ==========================================
if __name__ == "__main__":
    # 1. Gerar Dados
    gen = matriz_sudoku()
    x_raw, y_raw = gen.create_dataset(num_samples=10000, min_clues=4, max_clues=8)

    # 2. Separar em Treino e Teste e Codificar
    split_idx = 9000
    x_train_raw, x_test_raw = x_raw[:split_idx], x_raw[split_idx:]
    y_train_raw, y_test_raw = y_raw[:split_idx], y_raw[split_idx:]

    X_train, Y_train = one_hot_encode(x_train_raw, y_train_raw)
    X_test, Y_test = one_hot_encode(x_test_raw, y_test_raw)

    # 3. Inicializar e Treinar a RNA
    sudoku_rna = modelo_sudoku()
    sudoku_rna.train(X_train, Y_train, epochs=15)

    print("\n[AVALIAÇÃO DO MODELO]")

    loss, acc = sudoku_rna.model.evaluate(
        X_test,
        Y_test,
        verbose=0
    )

    print(f"Loss: {loss:.4f}")
    print(f"Acurácia: {acc:.4f}")

    # 4. Testar a Rede com Tabuleiros Iniciais Aleatórios
    print("\n[RESULTADOS DO TESTE]")
    test_samples = 2
    for i in range(test_samples):
        sample_x = X_test[i:i+1] # Pega um tabuleiro de teste

        # Prever com a RNA treinada
        pred_board = sudoku_rna.predict(sample_x)[0]

        print_board(x_test_raw[i], f"Entrada (Problema {i+1})")
        print_board(pred_board, f"Previsão da RNA (Problema {i+1})")
        print_board(y_test_raw[i], f"Gabarito Real (Problema {i+1})")

        # Etapa de Validação se a solução preenchida obedeceu às regras visuais
        is_correct = np.array_equal(
            pred_board,
            y_test_raw[i]
        )

        is_valid = validate_sudoku(
            pred_board
        )

        print(
            f"RNA Acertou exatamente o gabarito? "
            f"{'SIM' if is_correct else 'NÃO'}"
        )

        print(
            f"Solução respeita as regras do Sudoku? "
            f"{'SIM' if is_valid else 'NÃO'}"
        )

        print("=" * 40)

# ==========================================
# DISCUSSÃO SOBRE GENERALIZAÇÃO PARA NxN
# ==========================================

"""
GENERALIZAÇÃO DA SOLUÇÃO

Esta implementação foi desenvolvida especificamente para Sudoku 4x4,
com subgrupos 2x2 e conjunto S={1,2,3,4}.

Para generalizar a solução para um Sudoku NxN seriam necessárias
as seguintes modificações:

1) DIMENSÃO DO TABULEIRO

Atualmente:
    N = 4

Para generalização:
    N = tamanho do tabuleiro

Exemplos:
    4x4
    9x9
    16x16

---------------------------------------------------

2) TAMANHO DOS SUBGRUPOS

Atualmente:
    BOX = 2

Para generalização:

    import math
    BOX = int(math.sqrt(N))

Exemplos:

    N = 4  -> BOX = 2
    N = 9  -> BOX = 3
    N = 16 -> BOX = 4

---------------------------------------------------

3) REPRESENTAÇÃO DA ENTRADA

Atualmente:

    16 células

pois:

    4 x 4 = 16

Para NxN:

    N² células

Exemplos:

    4x4  -> 16 entradas
    9x9  -> 81 entradas
    16x16 -> 256 entradas

---------------------------------------------------

4) REPRESENTAÇÃO DA SAÍDA

Atualmente:

    16 células x 4 possibilidades

Para NxN:

    N² células x N possibilidades

Exemplos:

    4x4:
        16 x 4 = 64 saídas

    9x9:
        81 x 9 = 729 saídas

---------------------------------------------------

5) DIFICULDADES DA GENERALIZAÇÃO

a) Crescimento da dimensionalidade

Quanto maior o tabuleiro,
maior o número de neurônios necessários.

---------------------------------------------------

b) Necessidade de mais exemplos

A rede neural precisará de um conjunto
de treinamento muito maior para aprender
Sudokus maiores.

---------------------------------------------------

c) Tempo de treinamento

O treinamento aumenta significativamente
para Sudoku 9x9 e 16x16.

---------------------------------------------------

d) A rede aprende padrões e não regras

A RNA aprende relações estatísticas,
mas não compreende explicitamente
as regras do Sudoku.

Por isso ainda é necessária uma etapa
de validação da solução produzida.

---------------------------------------------------

e) Necessidade de re-treinamento

Uma rede treinada em Sudoku 4x4
não pode ser utilizada diretamente
em Sudoku 9x9.

A arquitetura de entrada e saída muda,
exigindo novo treinamento.
"""



/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iniciando treinamento da RNA...
Epoch 1/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8503 - loss: 0.4316 - val_accuracy: 0.9717 - val_loss: 0.0768
Epoch 2/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9812 - loss: 0.0529 - val_accuracy: 0.9762 - val_loss: 0.0589
Epoch 3/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9849 - loss: 0.0399 - val_accuracy: 0.9761 - val_loss: 0.0573
Epoch 4/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9860 - loss: 0.0370 - val_accuracy: 0.9786 - val_loss: 0.0561
Epoch 5/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9870 - loss: 0.0325 - val_accuracy: 0.9774 - val_loss: 0.0576
Epoch 6/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9882 - loss: 0.0302 - val_accuracy: 0.9763 - val_loss: 0.0580
Epoch 7/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9897 - loss: 0.0262 - val_accuracy: 0.9787 - val_loss: 0.0540
Epoch 8/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9894 -

'\nGENERALIZAÇÃO DA SOLUÇÃO\n\nEsta implementação foi desenvolvida especificamente para Sudoku 4x4,\ncom subgrupos 2x2 e conjunto S={1,2,3,4}.\n\nPara generalizar a solução para um Sudoku NxN seriam necessárias\nas seguintes modificações:\n\n1) DIMENSÃO DO TABULEIRO\n\nAtualmente:\n    N = 4\n\nPara generalização:\n    N = tamanho do tabuleiro\n\nExemplos:\n    4x4\n    9x9\n    16x16\n\n---------------------------------------------------\n\n2) TAMANHO DOS SUBGRUPOS\n\nAtualmente:\n    BOX = 2\n\nPara generalização:\n\n    import math\n    BOX = int(math.sqrt(N))\n\nExemplos:\n\n    N = 4  -> BOX = 2\n    N = 9  -> BOX = 3\n    N = 16 -> BOX = 4\n\n---------------------------------------------------\n\n3) REPRESENTAÇÃO DA ENTRADA\n\nAtualmente:\n\n    16 células\n\npois:\n\n    4 x 4 = 16\n\nPara NxN:\n\n    N² células\n\nExemplos:\n\n    4x4  -> 16 entradas\n    9x9  -> 81 entradas\n    16x16 -> 256 entradas\n\n---------------------------------------------------\n\n4) REPRESENTAÇÃO DA